In [ ]:
import copy
import torch
import numpy as np
import rasterio
import pandas as pd
import geopandas as gpd
import xgboost as xgb
import matplotlib.pyplot as plt
import torch.nn as nn
import kornia.augmentation as K
from pathlib import Path
from torch.utils.data import DataLoader
from torchgeo.datasets import RasterDataset, stack_samples, unbind_samples
from torchgeo.samplers import RandomGeoSampler, Units
from tqdm import tqdm
from pathlib import Path
from maw.preprocessing.augmentation import ClampAndRescale
from maw.datasets.rasters import Landsat, SpatialCovariates
from maw.models.resnet import build_resnet50
from importlib import import_module
from maw.utils.samplers import PointGeoSampler
from maw.training.metrics import compute_metrics
from maw.datasets.covariates import EXPECTED_COVARIATES

# Evaluate Model Performance using Census Data

In [ ]:
CKPT_STEM = "RESNET50-LANDSAT-BESTPARAMS-LR0.001-WD0.0001-DP0.3"
MODEL_CONFIG_PATH = "maw.configs.landsat_resnet50"
PREDS_STEM = "phl-resnet50-best-resnet-model"
ASSET_WEALTH_STEM = "phl_census_wealth_adm4"
N_FOLDS = 5   # number of spatial CV folds; must match mod.cfg.n_folds in maw.configs.landsat_resnet50

COVARIATES_STEM = "covariates-phl"
XGB_MODEL_STEM = "LANDSAT-XGBOOST-BESTPARAMS"
XGB_MODELS = {
    "spatial": f"{XGB_MODEL_STEM}-SPATIAL-ONLY",
    "embed": f"{XGB_MODEL_STEM}-EMBEDDINGS-ONLY",
    "all": f"{XGB_MODEL_STEM}-ALL-FEATURES",
}

In [ ]:
PATCH_FOOTPRINT_M = 6720          # 224 px * 30m Landsat resolution

SEED = 1234
IMAGERY_YEARS = [2010, 2020]
NUM_SAMPLES = 250000
IMAGERY_CRS = "EPSG:32651"
REBUILD_PTS = False

In [ ]:
HOME_DIR = Path.home() / "Projects/mapping-asia-wealth/"
ADM0_PATH = HOME_DIR / "data-raw/boundaries/ADM0/phl_adm0.geojson"
POP_PATH = HOME_DIR / "data-raw/census/PHL/phl_pop_2025_CN_100m_R2025A_v1.tif"
IMAGERY_DIR = HOME_DIR / "data-raw/imagery/country/cogs/"
LABELS_DIR = HOME_DIR / "data-raw/census/PHL"
PREDICTIONS_OUTDIR = HOME_DIR / "data-clean/outputs/model_predictions/country/"
LABELS_VCT_PATH = HOME_DIR / "data-raw/census/PHL/census_wealth_panel_admin4_spatial.gpkg"
XGB_MODEL_DIR = HOME_DIR / "data-clean/models/xgboost/"

In [ ]:
PREDICTIONS_OUTDIR.mkdir(exist_ok=True, parents=True)

In [ ]:
if REBUILD_PTS:

    # Randomly sample points, proportional to population from pop raster
    rng = np.random.default_rng(SEED)
    with rasterio.open(POP_PATH) as src:
        pop = src.read(1, masked=True)
        transform = src.transform
        src_crs = src.crs
    
        if src_crs is None:
            raise ValueError("Raster has no CRS. Define the source CRS before sampling.")
    
        # Keep valid, strictly positive population cells.
        weights = np.asarray(pop.filled(0), dtype="float64")
        valid = np.isfinite(weights) & (weights > 0)
    
        if not valid.any():
            raise ValueError("No positive population cells found.")
    
        rows, cols = np.nonzero(valid)
        w = weights[valid]
        p = w / w.sum(dtype="float64")
    
        # Sample cells with replacement, probability proportional to population.
        chosen = rng.choice(len(w), size=NUM_SAMPLES, replace=True, p=p)
        sampled_rows = rows[chosen]
        sampled_cols = cols[chosen]
    
        # Jitter uniformly within each selected raster cell.
        # Pixel coordinates: col + u, row + v, with u,v in [0, 1).
        u = rng.random(NUM_SAMPLES)
        v = rng.random(NUM_SAMPLES)
    
        x = (
            transform.a * (sampled_cols + u)
            + transform.b * (sampled_rows + v)
            + transform.c
        )
        y = (
            transform.d * (sampled_cols + u)
            + transform.e * (sampled_rows + v)
            + transform.f
        )
    
    weighted_pts = gpd.GeoDataFrame(
        {
            "point_id": np.arange(NUM_SAMPLES).astype(str),
            "pop_weight": w[chosen],
        },
        geometry=gpd.points_from_xy(x, y, crs=src_crs),
    )
    

    # Save
    weighted_pts = weighted_pts.to_crs(IMAGERY_CRS)
    weighted_pts.to_file(PREDICTIONS_OUTDIR / "phl-weighted-pts.gpkg")

In [ ]:
if not REBUILD_PTS:
    weighted_pts = gpd.read_file(PREDICTIONS_OUTDIR / "phl-weighted-pts.gpkg")

In [ ]:
# Plot
weighted_pts.plot(markersize=1)

## Extract Features and Run Model Inference (CNN + XGBoost) at Sample Points

This notebook samples a random set of points within the Phillippines, where the probability of selecting a point depends on the share of the population in the viscinity of that point. It then extracts Landsat CNN predictions + embeddings at those points. Finally, it extract spatial covariate values at those points, and runs inference using XGBoost models using the extracted embeddings and spatial covariates.

In [ ]:
# Restrict each point set to points with a corresponding census-derived AWI label for that year.
FILTERED_POINTS = {}
for imagery_year in IMAGERY_YEARS:
    label_bounds = gpd.read_file(LABELS_VCT_PATH).query(f'year == {imagery_year}').to_crs(IMAGERY_CRS)
    intersecting_idx = gpd.sjoin(
        weighted_pts,
        label_bounds[['geometry']],
        how='inner',
        predicate='intersects'
    ).index.unique()
    year_pts = weighted_pts.loc[intersecting_idx].reset_index(drop=True)
    FILTERED_POINTS[imagery_year] = year_pts
    print(f"Year: {imagery_year}   N: {len(weighted_pts)}")

## Run Landsat CNN Inference + Extract Embeddings


In [ ]:
PATCH_SIZE_PX = 224   # CNN input size, in pixels

for imagery_year in IMAGERY_YEARS:

    class AssetWealthDataset(RasterDataset):
        filename_glob = f'{ASSET_WEALTH_STEM}_{imagery_year}_common.tif'
        filename_regex = f'{ASSET_WEALTH_STEM}_(?P<date>\\d{{4}})'
        date_format = '%Y'
        is_image = False
        dtype = torch.float32

    landsat_dataset = Landsat(
        IMAGERY_DIR,
        filename_glob=f'landsat-phl-{imagery_year}.tif',
        filename_regex=r'landsat-phl-(?P<date>\d{4})',
    )
    awi_labels = AssetWealthDataset(LABELS_DIR)
    dataset = landsat_dataset & awi_labels

    sample_pts = FILTERED_POINTS[imagery_year]
    pts = tuple(zip(sample_pts.geometry.x, sample_pts.geometry.y))
    point_ids = sample_pts['point_id'].to_numpy()

    geo_sampler = PointGeoSampler(dataset=dataset, points_xy=pts, points_crs=IMAGERY_CRS, size=PATCH_SIZE_PX)
    assert len(geo_sampler) == len(sample_pts), \
        f"landsat/AWI sampler dropped points: {len(geo_sampler)} vs {len(sample_pts)}"
    landsat_loader = DataLoader(dataset, sampler=geo_sampler, batch_size=64, collate_fn=stack_samples,
                                prefetch_factor=1, num_workers=7, shuffle=False)

    mod = import_module(MODEL_CONFIG_PATH)
    transforms = mod.cfg.transforms_val.to(mod.cfg.device)

    model_predictions_list = []
    for held_out in range(1, N_FOLDS + 1):

        # Rebuilt fresh each fold: load_state_dict needs an unmodified Sequential `fc`, and
        # swapping it to nn.Identity below (to expose the backbone's embedding) would break
        # loading the next fold's checkpoint otherwise.
        ckpt = f"{CKPT_STEM}-SPLIT{held_out}-FINAL.pt"
        model = build_resnet50(hidden_dim=mod.cfg.fc_hidden, dropout=mod.cfg.dropout, frozen_backbone=mod.cfg.freeze_backbone, weights=mod.cfg.initial_weights)
        model = model.to(mod.cfg.device)
        model.load_state_dict(torch.load(mod.cfg.checkpoint_dir / ckpt, map_location=mod.cfg.device))
        embed_dim = model.fc[0].in_features

        # Split model into backbone + head so we can extract both the CNN's embedding and its
        # prediction in a single forward pass (as in extract_training_embeddings.ipynb)
        head = copy.deepcopy(model.fc)
        model.fc = torch.nn.Identity()
        backbone = model
        backbone.eval()
        head.eval()

        feats_list, preds_list, labels_list = [], [], []
        with torch.no_grad():
            for batch in tqdm(landsat_loader, desc=f"{imagery_year}-sampled-weighted-M{held_out}"):

                # Calculate AWI in patch footprint
                awi_patch = batch["mask"]
                awi_mask = awi_patch == -9999
                awi_masked = awi_patch.float().masked_fill(awi_mask, float('nan'))
                awi = awi_masked.nanmean(dim=(1, 2)).to(mod.cfg.device)

                # Extract embedding and predict asset wealth
                images = transforms(batch["image"].to(mod.cfg.device))
                feats = torch.flatten(backbone(images), start_dim=1)
                preds = head(feats)

                feats_list.append(feats.cpu())
                preds_list.append(preds.cpu())
                labels_list.append(awi.cpu())

        labels_arr = torch.cat(labels_list, dim=0).numpy().flatten()
        embeds_arr = torch.cat(feats_list, dim=0).numpy()
        outputs_arr = torch.cat(preds_list, dim=0).numpy().flatten()

        # Save CNN predictions
        meta = {
            "model_backbone": "resnet50-cnn",
            "model_backbone_name": ckpt,
            "model_backbone_heldoutset": held_out,
        }
        pred_df = sample_pts.copy()
        pred_df["labels"] = labels_arr
        pred_df["outputs"] = outputs_arr
        pred_df = pred_df.assign(**meta)
        model_predictions_list.append(pred_df)
        pred_df.to_parquet(PREDICTIONS_OUTDIR / f"{PREDS_STEM}-{imagery_year}-sampled-weighted-M{held_out}.parquet")

        preds_nona = pred_df.dropna(subset=['outputs', 'labels'])
        test_mae, test_rmse, test_r2 = compute_metrics(y_true=preds_nona['labels'].to_numpy(), y_pred=preds_nona['outputs'].to_numpy())
        print("CNN", imagery_year, held_out, {"test_mae": test_mae, "test_rmse": test_rmse, "test_r2": test_r2})

        # Save embeddings (+ labels + point_id) for downstream XGBoost inference. Saved
        # separately from pred_df/covariates so this step -- and the covariate extraction and
        # XGBoost inference steps below -- can each be rerun independently.
        embed_df = pd.DataFrame(embeds_arr, columns=[f"X{i}" for i in range(embed_dim)])
        embed_df.insert(0, "point_id", point_ids)
        embed_df["labels"] = labels_arr
        embed_df["model_backbone_heldoutset"] = held_out
        embed_df.to_parquet(PREDICTIONS_OUTDIR / f"{PREDS_STEM}-embeddings-{imagery_year}-sampled-weighted-M{held_out}.parquet")

    all_preds = pd.concat(model_predictions_list)
    all_preds.to_parquet(PREDICTIONS_OUTDIR / f"{PREDS_STEM}-{imagery_year}-sampled-weighted.parquet")

## Run XGBoost Inference

In [ ]:
# Extract Spatial Covariate Values

for imagery_year in IMAGERY_YEARS:

    covariates_raster = SpatialCovariates(
        IMAGERY_DIR,
        filename_glob=f'{COVARIATES_STEM}-{imagery_year}.tif',
        filename_regex=f'{COVARIATES_STEM}-(?P<date>\\d{{4}})',
    )
    sample_pts = FILTERED_POINTS[imagery_year]
    pts = tuple(zip(sample_pts.geometry.x, sample_pts.geometry.y))
    point_ids = sample_pts['point_id'].to_numpy()

    # Average covariate values over the same 6.72x6.72km footprint as the Landsat patch, sized
    # directly in CRS units so it doesn't depend on the covariates raster's own resolution.
    covs_sampler = PointGeoSampler(dataset=covariates_raster, points_xy=pts, points_crs=IMAGERY_CRS,
                                   size=PATCH_FOOTPRINT_M, units=Units.CRS)
    assert len(covs_sampler) == len(sample_pts), \
        f"covariates sampler dropped points: {len(covs_sampler)} vs {len(sample_pts)}"
    covs_loader = DataLoader(covariates_raster, sampler=covs_sampler, batch_size=64, collate_fn=stack_samples,
                             prefetch_factor=1, num_workers=7, shuffle=False)

    covs_list = []
    with torch.no_grad():
        for batch in tqdm(covs_loader, desc=f"Covariates {imagery_year}-sampled-weighted"):
            covs_patch = batch["mask"]                                  # (B, N_COVS, H, W)
            covs_nodata = covs_patch == -9999
            covs_masked = covs_patch.float().masked_fill(covs_nodata, float('nan'))
            covs_list.append(covs_masked.nanmean(dim=(2, 3)).cpu())     # (B, N_COVS)

    covs_arr = torch.cat(covs_list, dim=0).numpy()
    assert covs_arr.shape[1] == len(EXPECTED_COVARIATES), \
        f"Expected {len(EXPECTED_COVARIATES)} covariate bands, got {covs_arr.shape[1]}"

    covs_df = pd.DataFrame(covs_arr, columns=list(EXPECTED_COVARIATES))
    covs_df.insert(0, "point_id", point_ids)
    covs_df.to_parquet(PREDICTIONS_OUTDIR / f"{PREDS_STEM}-covariates-{imagery_year}-sampled-weighted.parquet")

    n_all_nodata = covs_df[list(EXPECTED_COVARIATES)].isna().all(axis=1).sum()
    print(f"Covariates {imagery_year}: N={len(covs_df)}, all-nodata points={n_all_nodata}")


In [ ]:
for imagery_year in IMAGERY_YEARS:

    sample_pts = FILTERED_POINTS[imagery_year]

    covs_df = pd.read_parquet(PREDICTIONS_OUTDIR / f"{PREDS_STEM}-covariates-{imagery_year}-sampled-weighted.parquet")
    covs_df["_all_nodata"] = covs_df[list(EXPECTED_COVARIATES)].isna().all(axis=1)

    xgb_predictions_list = {feature_type: [] for feature_type in XGB_MODELS}
    for held_out in range(1, N_FOLDS + 1):

        embed_df = pd.read_parquet(
            PREDICTIONS_OUTDIR / f"{PREDS_STEM}-embeddings-{imagery_year}-sampled-weighted-M{held_out}.parquet"
        )

        # Join this fold's embeddings to the covariates by point_id
        xgb_features = embed_df.merge(covs_df, on="point_id", how="inner", validate="one_to_one")
        assert len(xgb_features) == len(embed_df), (
            f"embeddings/covariates point_id mismatch: {len(embed_df)} embeddings "
            f"vs {len(xgb_features)} merged rows"
        )

        for feature_type, model_stem in XGB_MODELS.items():
            xgb_model = xgb.XGBRegressor()
            xgb_model.load_model(XGB_MODEL_DIR / f"{model_stem}-SPLIT{held_out}.json")
            feature_names = xgb_model.get_booster().feature_names
            xgb_outputs = xgb_model.predict(xgb_features[feature_names], validate_features=True)

            results = xgb_features[["point_id", "labels", "_all_nodata"]].copy()
            results["outputs"] = xgb_outputs

            xgb_pred_df = sample_pts.merge(results.drop(columns="_all_nodata"), on="point_id", how="inner")
            assert len(xgb_pred_df) == len(results), "sample_pts merge dropped rows"
            xgb_pred_df = xgb_pred_df.assign(
                model_backbone=f"xgboost-{feature_type}",
                model_backbone_name=f"{model_stem}-SPLIT{held_out}.json",
                model_backbone_heldoutset=held_out,
            )
            xgb_predictions_list[feature_type].append(xgb_pred_df)
            xgb_pred_df.to_parquet(
                PREDICTIONS_OUTDIR / f"{PREDS_STEM}-xgb-{feature_type}-{imagery_year}-sampled-weighted-M{held_out}.parquet"
            )

            xgb_preds_nona = xgb_pred_df.dropna(subset=['outputs', 'labels'])
            xgb_mae, xgb_rmse, xgb_r2 = compute_metrics(y_true=xgb_preds_nona['labels'].to_numpy(), y_pred=xgb_preds_nona['outputs'].to_numpy())
            print(f"XGB-{feature_type}", imagery_year, held_out, {"test_mae": xgb_mae, "test_rmse": xgb_rmse, "test_r2": xgb_r2})

    for feature_type, preds_list_ft in xgb_predictions_list.items():
        all_xgb_preds = pd.concat(preds_list_ft)
        all_xgb_preds.to_parquet(PREDICTIONS_OUTDIR / f"{PREDS_STEM}-xgb-{feature_type}-{imagery_year}-sampled-weighted.parquet")